# SurgIQ — YOLOv8 Training on Google Colab

This notebook trains the YOLOv8n surgical instrument detector on Google Colab free tier (T4 GPU).

**Steps:**
1. Mount Google Drive
2. Clone the SurgIQ GitHub repo
3. Install dependencies
4. Upload / verify dataset
5. Fine-tune YOLOv8n (50 epochs)
6. Evaluate on validation set
7. Save `best.pt` to Google Drive

**Before running:** Make sure `data/processed/yolo_dataset/` is prepared locally
and zipped as `yolo_dataset.zip`, then upload to your Google Drive.

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch Runtime to GPU!')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Cell 3 — Clone SurgIQ Repo

In [ ]:
import os

REPO_URL = 'https://github.com/DEKU-12/Surgiq.git'  # your repo
REPO_DIR = '/content/surgiq'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 4 — Install Dependencies

In [ ]:
!pip install ultralytics torch torchvision tqdm pyyaml --quiet
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')

## Cell 5 — Upload Dataset from Drive

Upload `yolo_dataset.zip` to `/MyDrive/surgiq/` in Google Drive first.

To create it locally:
```bash
cd data/processed
zip -r yolo_dataset.zip yolo_dataset/
```

In [ ]:
import os

DRIVE_ZIP = '/content/drive/MyDrive/surgiq/yolo_dataset.zip'
LOCAL_DIR = '/content/surgiq/data/processed'

os.makedirs(LOCAL_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    !unzip -q {DRIVE_ZIP} -d {LOCAL_DIR}
    print('Dataset extracted successfully.')
    !ls {LOCAL_DIR}/yolo_dataset/
else:
    print(f'ERROR: {DRIVE_ZIP} not found.')
    print('Please upload yolo_dataset.zip to your Google Drive at /MyDrive/surgiq/')

## Cell 6 — Fine-tune YOLOv8n (50 epochs)

In [ ]:
from ultralytics import YOLO
import yaml

DATASET_YAML = '/content/surgiq/data/processed/yolo_dataset/dataset.yaml'
OUTPUT_DIR   = '/content/surgiq/models/instrument_detector'

# Patch dataset.yaml path to absolute Colab paths
with open(DATASET_YAML, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/surgiq/data/processed/yolo_dataset'
with open(DATASET_YAML, 'w') as f:
    yaml.dump(data, f)

print('Dataset YAML updated for Colab paths.')
print(yaml.dump(data))

In [ ]:
model = YOLO('yolov8n.pt')  # Downloads ~6MB base model

results = model.train(
    data        = DATASET_YAML,
    epochs      = 50,
    batch       = 16,
    imgsz       = 640,
    device      = 0,          # T4 GPU
    patience    = 10,
    save        = True,
    save_period = 5,
    project     = OUTPUT_DIR,
    name        = 'train',
    exist_ok    = True,
    verbose     = True,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    flipud      = 0.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
)

print('\nTraining complete.')
print(f'Best mAP@0.5: {results.results_dict["metrics/mAP50(B)"]:.4f}')

## Cell 7 — Evaluate on Validation Set

In [ ]:
best_weights = f'{OUTPUT_DIR}/train/weights/best.pt'
model = YOLO(best_weights)

val_results = model.val(
    data   = DATASET_YAML,
    split  = 'val',
    imgsz  = 640,
    device = 0,
)

print(f'\nmAP@0.5      : {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {val_results.box.map:.4f}')
print(f'Precision    : {val_results.box.mp:.4f}')
print(f'Recall       : {val_results.box.mr:.4f}')

## Cell 8 — Save best.pt to Google Drive

In [ ]:
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/surgiq/models/'
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy2(best_weights, f'{DRIVE_OUT}/best.pt')
print(f'Saved best.pt to Google Drive: {DRIVE_OUT}/best.pt')
print('\nNext steps:')
print('  1. Download best.pt from Drive to your Mac')
print('  2. Place it at: models/instrument_detector/best.pt')
print('  3. Upload to GitHub Releases as v0.1-models')